# Replacing Transformer Attention with a Linear Running State

**Question:** what happens if we take a standard decoder-only Transformer,
delete the self-attention layer entirely, and replace it with a *moving
linear equation* that compresses all past tokens into a **fixed-size
running state**?

```
h_t = d * h_{t-1} + (1 - d) * (B x_t)      <- linear recurrence
y_t = C h_t                                 <- readout
```

- `d` in (0,1): learned per-channel decay (how fast the past fades)
- `B`, `C`: learned projections in/out of the state
- `h`: **fixed size** — it never grows with context length

Complexity per layer: attention is **O(T²)** time / **O(T)** memory,
the running state is **O(T)** time / **O(1)** memory.

**Experiment plan:** train two *identical* tiny LMs on 2–10 MB of text —
one with causal self-attention (baseline), one with the linear running
state — then compare loss, perplexity, speed, and generated samples.


## 1. Setup — GPU check

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"


## 2. Data — build a ~2–10 MB text corpus

TinyShakespeare (~1 MB) plus three public-domain Gutenberg books
(Alice in Wonderland, Frankenstein, Pride & Prejudice) lands us
comfortably inside the 2–10 MB window.


In [ ]:
import os, urllib.request

os.makedirs("data", exist_ok=True)
URLS = {
    "tinyshakespeare.txt": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "alice.txt":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "frankenstein.txt":    "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "pride.txt":           "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
}
parts = []
for name, url in URLS.items():
    p = f"data/{name}"
    if not os.path.exists(p):
        print("downloading", name)
        urllib.request.urlretrieve(url, p)
    with open(p, encoding="utf-8", errors="ignore") as f:
        parts.append(f.read())
text = "\n".join(parts)
print(f"corpus size: {len(text)/1e6:.2f} MB  ({len(text)} chars)")


## 3. Tokenizer + batches (character-level, keeps it simple)

In [ ]:
import torch

chars = sorted(set(text))
vocab = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.95 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"vocab size: {vocab}, train chars: {len(train_data)}, val chars: {len(val_data)}")

SEQ_LEN, BATCH = 128, 32
def get_batch(split):
    d = train_data if split == "train" else val_data
    i = torch.randint(0, len(d) - SEQ_LEN - 1, (BATCH,))
    x = torch.stack([d[j:j+SEQ_LEN] for j in i]).to(device)
    y = torch.stack([d[j+1:j+SEQ_LEN+1] for j in i]).to(device)
    return x, y


## 4. The two models — identical except the mixing layer

`CausalSelfAttention` = the standard Transformer layer (baseline).
`LinearRunningState` = the experiment: attention deleted, replaced by the
fixed-size linear recurrence.


In [ ]:
"""
Linear-recurrent replacement for Transformer self-attention.

Experiment: take a standard decoder-only Transformer, rip out the
causal self-attention layer, and replace it with a *moving linear
equation* that compresses the past into a fixed-size running state:

    h_t = d * h_{t-1} + (1 - d) * (B x_t)        (linear recurrence)
    y_t = C h_t                                   (readout)

where d in (0,1)^S is a learned per-channel decay, B/C are learned
projections, and h has FIXED size S independent of sequence length.

This is O(T) time and O(1) memory w.r.t. context length, vs O(T^2)
for attention. The question: how much modeling power do we lose?
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Token mixing layers
# ----------------------------------------------------------------------------

class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention (the baseline)."""
    def __init__(self, dim, n_heads, dropout=0.0):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # (B,T,H,D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B,H,T,D)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)


class LinearRunningState(nn.Module):
    """Attention replacement: fixed-size running state, linear recurrence.

    h_t = decay * h_{t-1} + (1 - decay) * (B x_t)
    y_t = C h_t

    - decay: learned per-state-channel forget factor in (0, 1)
    - B:     input  projection  dim -> state_dim
    - C:     output projection  state_dim -> dim
    State size is FIXED (does not grow with context length).
    """
    def __init__(self, dim, state_dim=None, dropout=0.0):
        super().__init__()
        self.state_dim = state_dim or dim
        # init decay ~ U(0.9, 0.999) -> long-ish memory at start
        init = torch.empty(self.state_dim).uniform_(math.log(0.9 / 0.1),
                                                   math.log(0.999 / 0.001))
        self.logit_decay = nn.Parameter(init)
        self.in_proj = nn.Linear(dim, self.state_dim, bias=False)
        self.out_proj = nn.Linear(self.state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        decay = torch.sigmoid(self.logit_decay)          # (S,) in (0,1)
        keep = 1.0 - decay
        u = self.in_proj(x)                              # (B,T,S)
        h = torch.zeros(B, self.state_dim, device=x.device, dtype=x.dtype)
        outs = []
        for t in range(T):
            h = decay * h + keep * u[:, t]
            outs.append(h)
        h_seq = torch.stack(outs, dim=1)                 # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


# ----------------------------------------------------------------------------
# Blocks & models (identical except for the mixing layer)
# ----------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, dim, n_heads, mixer, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.mix = mixer
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim), nn.GELU(),
            nn.Linear(4 * dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.mix(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyLM(nn.Module):
    """Decoder-only LM. mixer_fn(dim, n_heads) builds the token-mixing layer."""
    def __init__(self, vocab, dim=128, n_layers=4, n_heads=4,
                 seq_len=128, mixer_fn=None, dropout=0.0):
        super().__init__()
        self.seq_len = seq_len
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(
            torch.arange(T, device=idx.device))[None]
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temp=1.0, top_k=None):
        for _ in range(max_new):
            ctx = idx[:, -self.seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temp, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_models(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128,
                 state_dim=None, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout),
                  dropout=dropout)
    lin = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                 mixer_fn=lambda d, h: LinearRunningState(d, state_dim, dropout),
                 dropout=dropout)
    return attn, lin


## 5. Build both + parameter counts

In [ ]:
from models import build_models, count_params

DIM, LAYERS, HEADS = 128, 4, 4
attn, lin = build_models(vocab, dim=DIM, n_layers=LAYERS, n_heads=HEADS,
                         seq_len=SEQ_LEN)
models = {"attention (baseline)": attn.to(device),
          "linear running state": lin.to(device)}
for name, m in models.items():
    print(f"{name}: {count_params(m)/1e6:.3f}M params")

# sanity: shapes + one forward pass each
xb, yb = get_batch("train")
for name, m in models.items():
    logits, loss = m(xb, yb)
    print(name, "logits", tuple(logits.shape), "loss", round(float(loss), 3))


## 6. Train both side-by-side

Same optimizer, same LR, same batches — the *only* difference is the
mixing layer. (~5–10 min on a Colab T4 for 2000 steps.)


In [ ]:
import math, time

STEPS, LR, EVAL_EVERY = 2000, 3e-4, 200
opt = {n: torch.optim.AdamW(m.parameters(), lr=LR) for n, m in models.items()}
hist = {n: {"train": [], "val": [], "ms": []} for n in models}

for step in range(1, STEPS + 1):
    for name, m in models.items():
        m.train()
        xb, yb = get_batch("train")
        t0 = time.time()
        _, loss = m(xb, yb)
        opt[name].zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt[name].step()
        hist[name]["train"].append(float(loss))
        hist[name]["ms"].append((time.time() - t0) * 1000)
    if step % EVAL_EVERY == 0 or step == 1:
        msg = f"step {step:5d}"
        for name, m in models.items():
            m.eval()
            with torch.no_grad():
                vl = sum(float(m(*get_batch("val"))[1]) for _ in range(10)) / 10
            hist[name]["val"].append(vl)
            msg += (f" | {name}: train {hist[name]['train'][-1]:.3f} "
                    f"val {vl:.3f} (ppl {math.exp(vl):.1f})")
        print(msg, flush=True)

print("done.")
for name in models:
    ms = sum(hist[name]['ms']) / len(hist[name]['ms'])
    print(f"{name}: avg {ms:.1f} ms/batch")


## 7. Loss curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
for name in models:
    tr = hist[name]["train"]
    xs = range(1, len(tr) + 1)
    plt.plot(xs, tr, alpha=0.35, label=name + " train")
    ev = hist[name]["val"]
    ex = [i * EVAL_EVERY for i in range(1, len(ev) + 1)]
    plt.plot([1] + ex, [ev[0]] + ev, marker="o", label=name + " val")
plt.xlabel("step"); plt.ylabel("cross-entropy loss")
plt.legend(); plt.grid(alpha=0.3); plt.show()


## 8. Generate samples — can you tell which is which?

In [ ]:
for name, m in models.items():
    m.eval()
    ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
    out = m.generate(ctx, 300, temp=0.8, top_k=40)[0].tolist()
    print(f"----- {name} -----")
    print("".join(itos[i] for i in out))
    print()


## 9. Reading the results

**What to look for:**

| Signal | Meaning |
|---|---|
| Val loss / perplexity gap | How much modeling power the fixed state loses vs attention |
| ms/batch | The speed win of O(T) vs O(T²) — grows with `SEQ_LEN` |
| Sample quality | Whether the linear model keeps coherence over long spans |

**Expected outcome (hypothesis):** the linear state trains faster and
uses constant memory, but scores worse on perplexity — especially on
long-range dependencies (e.g. closing a quote opened 100 chars ago),
because a fixed-size vector is a lossy compression of the past while
attention keeps *every* past token around.

**Try next:**
1. Bump `SEQ_LEN` to 512 — the speed gap should widen, the quality gap too.
2. Increase the state size (`state_dim=256`) — does a bigger fixed memory close the gap?
3. Stack test: give the linear model *more layers* for equal wall-clock time — can depth compensate for the missing attention?
4. Inspect learned decays: `lin.blocks[0].mix.logit_decay.sigmoid()` — which channels learned long vs short memory?
